<a href="https://colab.research.google.com/github/Harshavardhan-gowdu-v/Harsha-POC-CODE/blob/main/extract_text_and_image_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pymupdf sentence-transformers faiss-cpu pillow matplotlib

In [ ]:
!pip install PyMuPDF PyPDF2

In [ ]:
from google.colab import files
import fitz  # PyMuPDF
from PyPDF2 import PdfReader

# Upload PDF
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

print("Uploaded:", file_path)

# ----------- Count Pages (Method 1: PyPDF2) -----------
reader = PdfReader(file_path)
num_pages = len(reader.pages)
print("📄 Total Pages:", num_pages)

# ----------- Count Images (Method 2: PyMuPDF) -----------
doc = fitz.open(file_path)

image_count = 0

for page_num in range(len(doc)):
    page = doc[page_num]
    images = page.get_images(full=True)
    image_count += len(images)

print("🖼️ Total Images:", image_count)

In [ ]:
import fitz  # PyMuPDF
import os

doc = fitz.open(file_path)

image_folder = "/content/pdf_images"
os.makedirs(image_folder, exist_ok=True)

texts = []
image_paths = []

for page_num in range(len(doc)):
    page = doc[page_num]

    # Extract text
    text = page.get_text()
    texts.append(text)

    # Extract images
    images = page.get_images(full=True)

    for img_index, img in enumerate(images):
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]

        img_path = f"{image_folder}/page{page_num}_img{img_index}.png"

        with open(img_path, "wb") as f:
            f.write(image_bytes)

        image_paths.append((page_num, img_path))

print("✅ Extracted text and images")

In [ ]:
from sentence_transformers import SentenceTransformer
from PIL import Image
import numpy as np

# Load CLIP model (for images)
clip_model = SentenceTransformer('clip-ViT-B-32')

image_embeddings = []
image_files = []

print("🔄 Creating image embeddings...")

for page_num, img_path in image_paths:
    try:
        img = Image.open(img_path).convert("RGB")
        emb = clip_model.encode(img)

        image_embeddings.append(emb)
        image_files.append(img_path)
    except:
        continue

image_embeddings = np.array(image_embeddings)

print("✅ Image embeddings ready:", len(image_embeddings))

In [ ]:
print("🖼️ Extracted Images Count:", len(image_paths))

🖼️ Extracted Images Count: 255


In [ ]:
from IPython.display import display, Image

# Print sample text (first page or first text block)
print("📄 Sample Text:\n")
print(texts[0][:1000])  # first 1000 characters

print("\n🖼️ First 3 Extracted Images:\n")

# Show first 3 images
for i, (page_num, img_path) in enumerate(image_paths[:30]):
    print(f"Image {i+1} (Page {page_num}):")
    display(Image(filename=img_path))

In [ ]:
chunks = []
chunk_page_map = []

for i, text in enumerate(texts):
    sentences = text.split(". ")

    chunk = ""
    for sent in sentences:
        chunk += sent + ". "

        if len(chunk) > 500:
            chunks.append(chunk)
            chunk_page_map.append(i)
            chunk = ""

# Count chunks
print("🧩 Total Chunks:", len(chunks))

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(chunks)

index = faiss.IndexFlatL2(len(embeddings[0]))
index.add(np.array(embeddings))

print("✅ Embeddings ready")

In [ ]:
def get_image_for_page(page_num):
    for p, path in image_paths:
        if p == page_num:
            return path
    return None

In [ ]:
from IPython.display import display, Image as IPImage
import numpy as np

def chatbot():
    print("🤖 Advanced PDF Chatbot Ready (Text + Images)\n")

    while True:
        try:
            query = input("🧑 Ask your question: ").strip()

            if query.lower() == "exit":
                print("👋 Exiting chatbot...")
                break

            if not query:
                print("⚠️ Enter a valid question\n")
                continue

            # 🔹 Step 1: Embed query
            query_vec = model.encode([query])

            # 🔹 Step 2: Retrieve top chunks
            D, I = index.search(np.array(query_vec), k=5)

            # 🔹 Step 3: Get relevant chunks + pages
            retrieved_chunks = [chunks[i] for i in I[0]]
            retrieved_pages = [chunk_page_map[i] for i in I[0]]

            # 🔹 Step 4: Combine answer text (clean + limited)
            answer_text = "\n\n".join(retrieved_chunks[:5])

            print("\n📄 Answer:\n")
            print(answer_text)

            # 🔥 Step 5: Get ALL relevant pages
            unique_pages = list(set(retrieved_pages))
            print(f"\n📌 Relevant Pages: {unique_pages}")

            # 🔥 Step 6: Collect ALL images from these pages
            all_relevant_images = []

            for page in unique_pages:
                imgs = [
                    img_path for pg, img_path in image_paths
                    if pg == page
                ]
                all_relevant_images.extend(imgs)

            # 🔥 Remove duplicates
            all_relevant_images = list(set(all_relevant_images))

            print("DEBUG → Total relevant images:", len(all_relevant_images))

            # 🔥 Step 7: Display images
            if all_relevant_images:
                print("\n🖼️ All Related Images:\n")

                for img_path in all_relevant_images:
                    try:
                        display(IPImage(filename=img_path))
                    except Exception as e:
                        print("⚠️ Error displaying image:", e)
            else:
                print("\n⚠️ No relevant images found")

            print("\n" + "="*60 + "\n")

        except Exception as e:
            print("⚠️ Error:", e)

In [ ]:
from IPython.display import display, Image as IPImage

display(IPImage(filename=img_path))

In [ ]:
chatbot()